# DriveSight — Entraînement YOLO11s

Ce notebook n'entraîne **que YOLO11s**. Il conserve le script original `src/detection/train_yolo.ipynb` intact. Exécuter après la préparation du dataset.

## 1. Vérification de l'environnement

In [1]:

from pathlib import Path
import os
import time
import torch
import yaml
from ultralytics import YOLO

# Chemins du projet DriveSight
REPO = Path(r"C:\Users\Kenza\DriveSight")
DATA_YAML = REPO / "data" / "yolo" / "data.yaml"
RUNS_DIR = REPO / "runs"

print("Dépôt trouvé :", REPO.is_dir())
print("Dataset trouvé :", DATA_YAML.is_file())
print("Dossier de sortie :", RUNS_DIR)

if not REPO.is_dir():
    raise FileNotFoundError(f"Dépôt introuvable : {REPO}")

if not DATA_YAML.is_file():
    raise FileNotFoundError(f"Dataset introuvable : {DATA_YAML}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA indisponible.")

os.chdir(REPO)

print("Dépôt :", REPO)
print("Dataset YAML :", DATA_YAML)
print("GPU :", torch.cuda.get_device_name(0))
print("PyTorch :", torch.__version__)


Dépôt trouvé : True
Dataset trouvé : True
Dossier de sortie : C:\Users\Kenza\DriveSight\runs
Dépôt : C:\Users\Kenza\DriveSight
Dataset YAML : C:\Users\Kenza\DriveSight\data\yolo\data.yaml
GPU : NVIDIA GeForce RTX 3050 6GB Laptop GPU
PyTorch : 2.14.1+cu130


## 2. Vérification de la configuration du dataset

In [2]:
config = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))

# Ultralytics utilise les chemins indiqués dans data.yaml.
# Vérifier les chemins résolus pour train/val/test sans modifier la configuration.
dataset_root = Path(config.get("path", DATA_YAML.parent))
if not dataset_root.is_absolute():
    dataset_root = (DATA_YAML.parent / dataset_root).resolve()

for split in ("train", "val", "test"):
    split_path = config.get(split)
    if split_path is None:
        print(f"{split}: non défini dans data.yaml")
        continue
    p = Path(split_path)
    if not p.is_absolute():
        p = dataset_root / p
    print(f"{split}: {p} | existe : {p.exists()}")
    if split in ("train", "val") and not p.exists():
        raise FileNotFoundError(f"Chemin {split} manquant : {p}")


path: C:/Users/Kenza/DriveSight/data/yolo
train: images/train
val: images/val
test: images/test
nc: 9
names:
  0: pedestrian
  1: rider
  2: car
  3: truck
  4: bus
  5: motorcycle
  6: bicycle
  7: traffic light
  8: traffic sign

train: C:\Users\Kenza\DriveSight\data\yolo\images\train | existe : True
val: C:\Users\Kenza\DriveSight\data\yolo\images\val | existe : True
test: C:\Users\Kenza\DriveSight\data\yolo\images\test | existe : True


## 3. Entraînement

In [3]:
MODEL_WEIGHTS = "yolo11s.pt"
RUN_NAME = "bdd_yolo11s"
print("Modèle :", MODEL_WEIGHTS)
print("Dossier de sortie :", RUNS_DIR / RUN_NAME)


Modèle : yolo11s.pt
Dossier de sortie : C:\Users\Kenza\DriveSight\runs\bdd_yolo11s


In [4]:
# Entraînement
# Les paramètres sont identiques entre les modèles pour rendre la comparaison plus équitable.
RUNS_DIR.mkdir(parents=True, exist_ok=True)
model = YOLO(MODEL_WEIGHTS)
start_time = time.perf_counter()

train_results = model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=4,
    seed=42,
    deterministic=True,
    workers=2,
    device=0,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    save=True,
    val=True
)

elapsed_seconds = time.perf_counter() - start_time
best_weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
last_weights = RUNS_DIR / RUN_NAME / "weights" / "last.pt"

print(f"Durée d'entraînement : {elapsed_seconds/60:.1f} minutes")
print("Meilleurs poids :", best_weights, "| existe :", best_weights.is_file())
print("Derniers poids :", last_weights, "| existe :", last_weights.is_file())

if not best_weights.is_file():
    raise FileNotFoundError("L'entraînement s'est terminé mais best.pt est absent.")

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


New https://pypi.org/project/ultralytics/8.4.191 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.190  Python-3.11.9 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Kenza\DriveSight\data\yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.

## 4. Sorties produites

Les poids et les graphiques sont enregistrés dans `runs/{run_name}/`. Ne lancez pas ce notebook si vous souhaitez seulement évaluer un modèle déjà entraîné.